<a href="https://colab.research.google.com/github/VinayaSharada/FinancialAnalyticsCourse/blob/main/PortfolioOpt/portfolio_data_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Portfolio Optimization - Synthetic Data Generator

This notebook generates realistic synthetic financial data for portfolio optimization analysis.

**Course**: Financial Analytics Masters Course  
**Topic**: Portfolio Optimization  
**Author**: AI Assistant  
**Date**: July 2025  

## 📚 Setup and Installation

In [ ]:
# Install required packages
!pip install pandas numpy matplotlib seaborn scipy faker -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)
Faker.seed(42)

print("✅ All packages installed and imported successfully!")

## 🏗️ Portfolio Data Generator Class

In [ ]:
class PortfolioDataGenerator:
    """
    Generates synthetic portfolio data for financial analysis.
    
    This class creates realistic financial time series data including:
    - Multiple asset returns
    - Market indicators
    - Risk-free rates
    - Asset metadata
    """
    
    def __init__(self, locale='en_IN', seed=42):
        """
        Initialize the data generator.
        
        Parameters:
        -----------
        locale : str, default 'en_IN'
            Locale for faker library
        seed : int, default 42
            Random seed for reproducibility
        """
        self.fake = Faker(locale)
        Faker.seed(seed)
        np.random.seed(seed)
        self.locale = locale
        
    def generate_asset_metadata(self, n_assets=20):
        """Generate metadata for financial assets."""
        sectors = ['Technology', 'Healthcare', 'Finance', 'Consumer Goods', 
                  'Energy', 'Utilities', 'Real Estate', 'Materials', 
                  'Telecommunications', 'Industrials']
        
        assets = []
        for i in range(n_assets):
            asset = {
                'symbol': f"STOCK_{i+1:02d}",
                'company_name': self.fake.company(),
                'sector': np.random.choice(sectors),
                'market_cap': np.random.lognormal(mean=10, sigma=1.5),  # in millions
                'listing_date': self.fake.date_between(start_date='-10y', end_date='-1y')
            }
            assets.append(asset)
            
        return pd.DataFrame(assets)
    
    def _get_sector_parameters(self, sector):
        """Get sector-specific financial parameters."""
        sector_params = {
            'Technology': {'expected_return': 0.12, 'volatility': 0.25},
            'Healthcare': {'expected_return': 0.10, 'volatility': 0.18},
            'Finance': {'expected_return': 0.09, 'volatility': 0.22},
            'Consumer Goods': {'expected_return': 0.08, 'volatility': 0.15},
            'Energy': {'expected_return': 0.07, 'volatility': 0.30},
            'Utilities': {'expected_return': 0.06, 'volatility': 0.12},
            'Real Estate': {'expected_return': 0.08, 'volatility': 0.20},
            'Materials': {'expected_return': 0.09, 'volatility': 0.25},
            'Telecommunications': {'expected_return': 0.07, 'volatility': 0.16},
            'Industrials': {'expected_return': 0.09, 'volatility': 0.19}
        }
        return sector_params.get(sector, {'expected_return': 0.08, 'volatility': 0.20})
    
    def generate_market_data(self, start_date, end_date, n_assets=20):
        """Generate synthetic market data with realistic price movements."""
        # Create date range
        date_range = pd.date_range(start=start_date, end=end_date, freq='D')
        date_range = date_range[date_range.dayofweek < 5]  # Remove weekends
        
        n_days = len(date_range)
        print(f"Generating data for {n_days} trading days and {n_assets} assets")
        
        # Generate asset metadata
        asset_metadata = self.generate_asset_metadata(n_assets)
        
        # Initialize price matrix
        data = []
        
        # Market parameters
        market_volatility = 0.15
        risk_free_rate = 0.06
        
        for idx, row in asset_metadata.iterrows():
            symbol = row['symbol']
            sector = row['sector']
            
            # Sector-specific parameters
            sector_params = self._get_sector_parameters(sector)
            
            # Generate price series using geometric Brownian motion
            initial_price = np.random.uniform(50, 500)
            mu = sector_params['expected_return']
            sigma = sector_params['volatility']
            
            # Generate daily returns
            dt = 1/252  # Daily time step
            shocks = np.random.normal(0, 1, n_days)
            returns = mu * dt + sigma * np.sqrt(dt) * shocks
            
            # Calculate prices
            prices = [initial_price]
            for i in range(1, n_days):
                price = prices[-1] * np.exp(returns[i])
                prices.append(price)
            
            # Create data for this asset
            for i, date in enumerate(date_range):
                data.append({
                    'date': date,
                    'symbol': symbol,
                    'company_name': row['company_name'],
                    'sector': sector,
                    'market_cap': row['market_cap'],
                    'price': round(prices[i], 2),
                    'daily_return': round(returns[i], 6) if i > 0 else 0,
                    'volume': np.random.randint(10000, 1000000),
                    'risk_free_rate': risk_free_rate / 252,  # Daily risk-free rate
                    'market_return': round(np.random.normal(0.08/252, market_volatility/np.sqrt(252)), 6)
                })
        
        df = pd.DataFrame(data)
        
        # Add some market-wide events (crashes, rallies)
        df = self._add_market_events(df, date_range)
        
        return df
    
    def _add_market_events(self, df, date_range):
        """Add market-wide events to make data more realistic."""
        try:
            # Add a few market crashes/rallies
            n_events = max(1, len(date_range) // 500)  # About 1 event per 2 years
            event_dates = np.random.choice(date_range[100:-100], n_events, replace=False)
            
            for event_date in event_dates:
                event_magnitude = np.random.uniform(-0.15, 0.10)  # -15% to +10%
                event_mask = df['date'] == event_date
                df.loc[event_mask, 'daily_return'] += event_magnitude
                df.loc[event_mask, 'market_return'] += event_magnitude
                
            return df
            
        except Exception as e:
            print(f"Warning: Error adding market events: {str(e)}")
            return df
    
    def generate_dataset(self, n_rows=10000, start_date='2020-01-01', end_date='2024-12-31', 
                        n_assets=20, filename='syntheticdata.csv'):
        """Generate complete synthetic dataset for portfolio optimization."""
        # Calculate date range to approximately match n_rows
        business_days_per_year = 252
        years_needed = n_rows / (n_assets * business_days_per_year)
        
        if years_needed > 5:
            print(f"Warning: Requested {n_rows} rows requires {years_needed:.1f} years of data. Adjusting...")
            years_needed = 5
            
        # Adjust end date if needed
        start_dt = pd.to_datetime(start_date)
        calculated_end = start_dt + timedelta(days=int(years_needed * 365))
        end_dt = min(pd.to_datetime(end_date), calculated_end)
        
        print(f"Generating data from {start_dt.date()} to {end_dt.date()}")
        print(f"Assets: {n_assets}, Expected rows: ~{int(years_needed * n_assets * business_days_per_year)}")
        
        # Generate the data
        df = self.generate_market_data(start_dt, end_dt, n_assets)
        
        # Save to CSV
        df.to_csv(filename, index=False)
        print(f"Dataset saved to {filename}")
        print(f"Final dataset shape: {df.shape}")
        print(f"Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"Assets: {df['symbol'].nunique()}")
        
        return df

print("✅ PortfolioDataGenerator class defined successfully!")

## 🎛️ Configuration Parameters

**Students**: Modify these parameters to explore different scenarios:

In [ ]:
# =============================================================================
# CONFIGURATION PARAMETERS - MODIFY THESE TO EXPLORE DIFFERENT SCENARIOS
# =============================================================================

# Dataset size parameters
N_ROWS = 10000          # Target number of rows
N_ASSETS = 20           # Number of assets (stocks)
START_DATE = '2020-01-01'  # Start date for data
END_DATE = '2024-12-31'    # End date for data

# File parameters
FILENAME = 'syntheticdata.csv'  # Output filename
RANDOM_SEED = 42                # For reproducibility

print("📊 Configuration:")
print(f"   • Target rows: {N_ROWS:,}")
print(f"   • Number of assets: {N_ASSETS}")
print(f"   • Date range: {START_DATE} to {END_DATE}")
print(f"   • Output file: {FILENAME}")
print(f"   • Random seed: {RANDOM_SEED}")

## 🏃‍♂️ Generate Synthetic Data

In [ ]:
# Initialize the data generator
generator = PortfolioDataGenerator(locale='en_IN', seed=RANDOM_SEED)

# Generate the dataset
print("🚀 Starting data generation...")
df = generator.generate_dataset(
    n_rows=N_ROWS,
    start_date=START_DATE,
    end_date=END_DATE,
    n_assets=N_ASSETS,
    filename=FILENAME
)

print("\n✅ Data generation completed successfully!")

## 📊 Data Exploration and Validation

In [ ]:
# Display basic information
print("📋 DATASET OVERVIEW")
print("="*50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
display(df.head())

print(f"\nLast 5 rows:")
display(df.tail())

print(f"\nData types:")
print(df.dtypes)

print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Summary statistics
print("📈 SUMMARY STATISTICS")
print("="*50)

print(f"\nAsset Information:")
print(f"• Number of unique assets: {df['symbol'].nunique()}")
print(f"• Assets: {sorted(df['symbol'].unique())}")

print(f"\nSector Distribution:")
sector_counts = df.groupby('symbol')['sector'].first().value_counts()
print(sector_counts)

print(f"\nDate Information:")
print(f"• Date range: {df['date'].min()} to {df['date'].max()}")
print(f"• Number of trading days: {df['date'].nunique()}")
print(f"• Total observations: {len(df)}")

print(f"\nReturn Statistics:")
print(df['daily_return'].describe())

## 📈 Data Visualization

In [ ]:
# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Portfolio Synthetic Data Analysis', fontsize=16, fontweight='bold')

# 1. Price evolution for a few assets
sample_assets = df['symbol'].unique()[:5]
for asset in sample_assets:
    asset_data = df[df['symbol'] == asset]
    axes[0, 0].plot(asset_data['date'], asset_data['price'], label=asset, alpha=0.8)
axes[0, 0].set_title('Price Evolution (Sample Assets)')
axes[0, 0].set_xlabel('Date')
axes[0, 0].set_ylabel('Price')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Return distribution
axes[0, 1].hist(df['daily_return'], bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].set_title('Distribution of Daily Returns')
axes[0, 1].set_xlabel('Daily Return')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].grid(True, alpha=0.3)

# 3. Sector distribution
sector_counts.plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Number of Assets by Sector')
axes[1, 0].set_xlabel('Sector')
axes[1, 0].set_ylabel('Number of Assets')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# 4. Market cap distribution
market_caps = df.groupby('symbol')['market_cap'].first()
axes[1, 1].hist(market_caps, bins=20, alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Market Capitalization Distribution')
axes[1, 1].set_xlabel('Market Cap (Millions)')
axes[1, 1].set_ylabel('Number of Assets')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Return correlation heatmap
returns_pivot = df.pivot(index='date', columns='symbol', values='daily_return')
correlation_matrix = returns_pivot.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.1)
plt.title('Asset Return Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n📊 Correlation Statistics:")
print(f"• Average correlation: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].mean():.3f}")
print(f"• Max correlation: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].max():.3f}")
print(f"• Min correlation: {correlation_matrix.values[np.triu_indices_from(correlation_matrix.values, k=1)].min():.3f}")

## ✅ Data Quality Validation

In [ ]:
print("🔍 DATA QUALITY VALIDATION")
print("="*50)

# Check for missing values
missing_data = df.isnull().sum()
if missing_data.sum() == 0:
    print("✅ No missing values found")
else:
    print("⚠️ Missing values detected:")
    print(missing_data[missing_data > 0])

# Check date consistency
date_check = df.groupby('date')['symbol'].nunique()
expected_assets = N_ASSETS
if (date_check == expected_assets).all():
    print(f"✅ Consistent number of assets ({expected_assets}) across all dates")
else:
    print("⚠️ Inconsistent number of assets across dates")
    problematic_dates = date_check[date_check != expected_assets]
    print(f"Problematic dates: {len(problematic_dates)}")

# Check return reasonableness
extreme_returns = df[(df['daily_return'] > 0.5) | (df['daily_return'] < -0.5)]
if len(extreme_returns) == 0:
    print("✅ No extreme returns (>50% or <-50%) found")
else:
    print(f"⚠️ {len(extreme_returns)} extreme return observations found")

# Check price positivity
negative_prices = df[df['price'] <= 0]
if len(negative_prices) == 0:
    print("✅ All prices are positive")
else:
    print(f"❌ {len(negative_prices)} negative or zero price observations found")

# Calculate some financial metrics for validation
annual_returns = returns_pivot.mean() * 252
annual_volatilities = returns_pivot.std() * np.sqrt(252)

print(f"\n📊 Financial Metrics Summary:")
print(f"• Annualized returns range: {annual_returns.min():.2%} to {annual_returns.max():.2%}")
print(f"• Annualized volatilities range: {annual_volatilities.min():.2%} to {annual_volatilities.max():.2%}")
print(f"• Average Sharpe ratio (assuming 6% risk-free): {((annual_returns - 0.06) / annual_volatilities).mean():.3f}")

print("\n✅ Data quality validation completed!")

## 💾 Download Generated Data

The synthetic data has been saved as CSV file. You can download it for use in other applications.

In [ ]:
# For Google Colab users - download the generated file
try:
    from google.colab import files
    print(f"📥 Downloading {FILENAME}...")
    files.download(FILENAME)
    print("✅ Download initiated!")
except ImportError:
    print(f"📁 File saved locally as: {FILENAME}")
    print("💡 If running locally, you can find the file in your current directory")

## 🎓 Student Exploration Guide

### 🔧 Parameter Tuning Exercises:

1. **Dataset Size**: 
   - Modify `N_ROWS` to generate larger/smaller datasets
   - Observe impact on processing time and memory usage

2. **Number of Assets**: 
   - Change `N_ASSETS` to simulate different portfolio sizes
   - Try 10, 50, 100 assets and see the effect on correlations

3. **Time Period**: 
   - Adjust `START_DATE` and `END_DATE`
   - Try different market periods (bull vs bear markets)

4. **Sector Parameters**: 
   - Modify the `_get_sector_parameters()` method
   - Change expected returns and volatilities for different sectors

5. **Market Events**: 
   - Adjust the frequency and magnitude of market events
   - Add specific event types (earnings announcements, economic shocks)

### 📊 Analysis Exercises:

1. **Risk-Return Analysis**:
   - Calculate Sharpe ratios for all assets
   - Identify the best and worst performing sectors

2. **Correlation Analysis**:
   - Find pairs of assets with highest/lowest correlations
   - Analyze sector-wise correlation patterns

3. **Time Series Analysis**:
   - Look for trends and patterns in price movements
   - Identify periods of high/low volatility

### 🏦 Banking Applications:

This synthetic data generator can be used by banks to:

1. **Stress Testing**: Generate scenarios for portfolio stress testing
2. **Model Validation**: Create datasets for backtesting trading strategies
3. **Training**: Provide realistic data for training quantitative analysts
4. **Product Development**: Test new investment products before launch
5. **Risk Management**: Simulate various market conditions for risk assessment

## 📚 Next Steps

1. **Portfolio Optimization**: Use this data with the portfolio optimization notebook
2. **Risk Analysis**: Implement Value-at-Risk and other risk metrics
3. **Backtesting**: Test investment strategies using this synthetic data
4. **Machine Learning**: Use for training predictive models

---

**📧 Questions or Issues?**  
This notebook is part of the Financial Analytics Masters Course.  
For support, please refer to the course materials or contact your instructor.